# PromptPotter: Pipeline Analysis

```
INITIALIZE -> S1 -> GROW/FILTER -> ANALYSIS -> [HUMAN]
```

This notebook demonstrates the first step of the HITL optimization loop:
1. Fetch dataset from TermNorm API
2. Run a simple LLM prompt on each item
3. Evaluate results against expected outputs
4. Human reviews the failures

## Cell 1: INITIALIZE - Fetch Dataset

In [ ]:
import sys
sys.path.insert(0, '..')  # Add parent dir to path for imports

import httpx
from api.services.dataset_loader import DatasetLoader

# Configuration
TERMNORM_URL = "http://127.0.0.1:8000"
EXPERIMENT_ID = "1_production_historical"
TRACES_FILE = "../data/traces_1_production_historical.json"

# Load dataset from saved traces
loader = DatasetLoader()
dataset = loader.load_from_traces_json(TRACES_FILE)

print(f"Loaded {len(dataset)} items from {dataset.source}")
print(f"Dataset name: {dataset.name}")

In [ ]:
# Preview dataset items
import pandas as pd

if dataset.items:
    preview_data = []
    for item in dataset.items[:10]:
        preview_data.append({
            "id": item.id[:20] + "..." if len(item.id) > 20 else item.id,
            "input": item.input.get("query", str(item.input)),
            "expected": item.expected_output.get("target", str(item.expected_output)) if item.expected_output else "N/A"
        })
    
    df = pd.DataFrame(preview_data)
    display(df)
else:
    print("No items in dataset")

In [ ]:
# Fetch reference candidates from TermNorm API
async def fetch_candidates():
    async with httpx.AsyncClient() as client:
        resp = await client.get(f"{TERMNORM_URL}/experiments/{EXPERIMENT_ID}/mappings")
        resp.raise_for_status()
        return resp.json()

mappings_data = await fetch_candidates()
candidates = mappings_data["candidates"]

print(f"Loaded {len(candidates)} unique candidates")
print(f"Total mappings: {mappings_data['total_mappings']}")
print(f"\nFirst 5 candidates:")
for c in candidates[:5]:
    print(f"  - {c[:60]}...")

In [ ]:
# Load and configure the workflow
from pathlib import Path
from api.core.workflow_runner import WorkflowRunner

# Load workflow from YAML
WORKFLOW_PATH = Path("../workflows/examples/research_rank.yaml")

runner = WorkflowRunner.from_yaml(WORKFLOW_PATH)

print(f"Loaded workflow: {runner.workflow.id}")
print(f"Steps: {[s.id for s in runner.workflow.steps]}")
print(f"Inputs: {list(runner.workflow.inputs.keys())}")
print(f"Outputs: {list(runner.workflow.outputs.keys())}")

In [ ]:
# Run the workflow on dataset items (limit for testing)
MAX_ITEMS = 5  # Set to None for all items

results = []
items_to_process = dataset.items[:MAX_ITEMS] if MAX_ITEMS else dataset.items

print(f"Processing {len(items_to_process)} items with {len(candidates)} candidates...")

for i, item in enumerate(items_to_process):
    query = item.input.get("query", str(item.input))
    
    # Workflow inputs
    workflow_inputs = {
        "query": query,
        "candidates": candidates  # The reference terms to rank against
    }
    
    try:
        # Execute workflow
        context = await runner.execute(workflow_inputs)
        
        # Extract outputs
        best_match = context.step_outputs.get("candidate_ranker", {}).get("top_candidate", "")
        confidence = context.step_outputs.get("candidate_ranker", {}).get("confidence", 0)
        
        results.append({
            "id": item.id,
            "input": query,
            "expected": item.expected_output.get("target") if item.expected_output else None,
            "actual": best_match,
            "confidence": confidence,
            "status": context.status,
            "error": context.error
        })
        
        print(f"[{i+1}/{len(items_to_process)}] {query[:25]}... -> {best_match[:35]}... (conf: {confidence:.2f})")
        
    except Exception as e:
        results.append({
            "id": item.id,
            "input": query,
            "expected": item.expected_output.get("target") if item.expected_output else None,
            "actual": None,
            "confidence": 0,
            "status": "error",
            "error": str(e)
        })
        print(f"[{i+1}/{len(items_to_process)}] ERROR: {query[:30]}... -> {e}")

print(f"\nProcessed {len(results)} items")

## Cell 3: GROW/FILTER - Classify Results

In [ ]:
# Separate successes from failures
def normalize_for_comparison(s):
    """Normalize string for comparison (lowercase, strip whitespace)"""
    if s is None:
        return ""
    return str(s).lower().strip()

successes = []
failures = []
errors = []
no_expected = []

for r in results:
    if r["error"]:
        errors.append(r)
    elif r["expected"] is None:
        no_expected.append(r)
    elif normalize_for_comparison(r["actual"]) == normalize_for_comparison(r["expected"]):
        successes.append(r)
    else:
        failures.append(r)

print(f"Successes: {len(successes)}")
print(f"Failures:  {len(failures)}")
print(f"Errors:    {len(errors)}")
print(f"No expected output: {len(no_expected)}")

## Cell 4: ANALYSIS - Metrics & Failure Analysis

In [ ]:
# Calculate metrics
total_with_expected = len(successes) + len(failures)
accuracy = len(successes) / total_with_expected if total_with_expected > 0 else 0

print("=" * 50)
print("ANALYSIS SUMMARY")
print("=" * 50)
print(f"Total items processed: {len(results)}")
print(f"Items with expected output: {total_with_expected}")
print(f"Accuracy: {accuracy:.1%}")
print(f"Error rate: {len(errors) / len(results):.1%}" if results else "N/A")
print("=" * 50)

In [ ]:
# Show failure cases for analysis
if failures:
    print("\nFAILURE CASES:")
    print("-" * 50)
    
    failure_df = pd.DataFrame(failures)
    display(failure_df[["input", "expected", "actual"]])
else:
    print("\nNo failures! All predictions matched expected outputs.")

In [ ]:
# Show error cases
if errors:
    print("\nERROR CASES:")
    print("-" * 50)
    
    error_df = pd.DataFrame(errors)
    display(error_df[["input", "error"]])

# Summary for human review
print("""
================================================================================
                           HUMAN REVIEW
================================================================================

Review the failure cases above. Consider:

1. Is the workflow appropriate for this task?
2. Are there patterns in the failures?
3. Would different web search or ranking parameters help?

Next steps (not yet implemented):
- [ ] Modify workflow parameters
- [ ] Try different LLM model
- [ ] Adjust candidate ranking prompt
- [ ] Test with/without web search

================================================================================
""")

# Store results for later analysis
analysis_results = {
    "accuracy": accuracy,
    "total": len(results),
    "successes": len(successes),
    "failures": len(failures),
    "errors": len(errors),
    "workflow": str(WORKFLOW_PATH),
    "num_candidates": len(candidates),
    "failure_cases": failures
}

print(f"Results stored in 'analysis_results' variable.")

In [ ]:
# Summary for human review
print("""
================================================================================
                           HUMAN REVIEW
================================================================================

Review the failure cases above. Consider:

1. Is the workflow appropriate for this task?
2. Are there patterns in the failures?
3. Would different web search or ranking parameters help?

Next steps (not yet implemented):
- [ ] Modify workflow parameters
- [ ] Try different LLM model
- [ ] Adjust candidate ranking prompt
- [ ] Test with/without web search

================================================================================
""")

# Store results for later analysis
analysis_results = {
    "accuracy": accuracy,
    "total": len(results),
    "successes": len(successes),
    "failures": len(failures),
    "errors": len(errors),
    "workflow": str(WORKFLOW_PATH),
    "num_candidates": len(candidates),
    "failure_cases": failures
}

print(f"Results stored in 'analysis_results' variable.")